In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]  # go up from notebooks/
sys.path.append(str(ROOT))

In [2]:
from config import SEED, N_LIST, METHODS, RESULTS_PATH
from models.cvae import train_cvae_on_arrays, sample_cvae_dataset
from models.bootstrap import sample_bootstrap
from models.gmm import sample_gmm
import numpy as np
import pandas as pd
from metrics import *
import ipywidgets as widgets
from IPython.display import display


# Loading Data
1. Breast cancer
2. Pima Indians Diabetes
3. Somethng else 3

#### 1. Breast Cancer

| Dataset | Category | n | p | Class 0 | Class 1 | Source | Notes |
|---------|----------|---|---|--------|--------|--------|-------|
| Breast Cancer | clinical_tabular | 569 | 30 | 212 | 357 | sklearn | moderate n, moderate p, numeric only |

#### 2. Diabetes

| Dataset | Category | n | p | Class 0 | Class 1 | Source | Notes |
|---------|----------|---|---|--------|--------|--------|-------|
| Diabetes| metabolic_tabular | 768 | 8 | 500 | 268 | openml | high n, low p, numeric only |

In [3]:
from loaders import load_breast, load_diabetes
data = load_diabetes()

X = data["X"]
y = data["y"]
feature_names = data["feature_names"]

print(data["dataset"], data["category"])
print(X.shape, y.shape)
print(np.bincount(y))

diabetes metabolic_tabular
(768, 8) (768,)
[500 268]


# Helper Functions
- Stratify Sampler

In [4]:
X_small, y_small, idx_small = stratified_subsample(X, y, n0=23, n1=68, seed=42)
print(X_small.shape)
print(np.bincount(y_small))

(91, 8)
[23 68]


- Train & sample CVAE

In [5]:

best_state = train_cvae_on_arrays(
    X,
    y,
    seed=42,
    z_dim=16,
    hidden=128,
    beta=0.5,
    lr=1e-3,
    epochs=200,
    batch_size=32,
)

X_syn, y_syn = sample_cvae_dataset(
    best_state,
    n0=23,
    n1=68,
    seed=42
)

print(X_syn.shape)
print(np.bincount(y_syn))

Epoch  50 | val loss=4.6199 recon=2.2393 kl=4.7613
Epoch 100 | val loss=4.4593 recon=2.1760 kl=4.5667
Epoch 150 | val loss=4.3230 recon=2.1170 kl=4.4120
Epoch 200 | val loss=4.2649 recon=2.0223 kl=4.4853
(91, 8)
[23 68]


- bootstrap

In [6]:
X_syn_b, y_syn_b = sample_bootstrap(X_small, y_small, n0=23, n1=68, seed=42)

- gmm

In [7]:
X_syn_g, y_syn_g = sample_gmm(X_small, y_small, n0=23, n1=68, seed=42, n_components=2)

## Selection Side
- `forward` means keep top k
- `reverse` means drop top k

In [8]:
def select_feature_subset(X, feature_names, mode="full", ranked_idx=None, k=None, drop_idx=None):
    p = X.shape[1]
    
    if mode == "full":
        keep = np.arange(p)

    elif mode == "drop_one":
        keep = np.array([j for j in range(p) if j != drop_idx])

    elif mode == "forward":
        keep = np.array(ranked_idx[:k])

    elif mode == "reverse":
        keep = np.array(ranked_idx[k:])

    else:
        raise ValueError(f"Unknown mode: {mode}")

    return keep

In [9]:
def rank_features_by_rf_importance(X, y, seed=42, n_estimators=15):
    rf = RandomForestClassifier(n_estimators=n_estimators, random_state=seed)
    rf.fit(X, y)
    return np.argsort(rf.feature_importances_)[::-1]

## Displaying Plots

In [10]:
def show_figures(figures, dataset=None, method=None, plot =None):
    for item in figures:
        if dataset is not None and item["dataset"] != dataset:
            continue
        if method is not None and item["method"] != method:
            continue

        fig_obj = item["fig"]

        if isinstance(fig_obj, dict):
            if plot is None:
                selected = fig_obj.items()
            else:
                if plot not in fig_obj:
                    continue
                selected = [(plot, fig_obj[plot])]

            for name, fig in selected:
                print(f"{item['dataset']} | {item['method']} | {name}")
                display(fig)
        else:
            print(f"{item['dataset']} | {item['method']}")
            display(fig_obj)

# The actual tests with numbers or something

In [ ]:
metrics = evaluate_all(X_real=X, X_syn=X_syn, y_real= y, y_syn = y_syn)
print(metrics)

### Results row

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

def run_experiment(n0, n1, feature_mode = "full", k = None, drop_idx = None, rows=[], figs = None):
    
    
    datasets = [
        load_breast,
        load_diabetes,
    ]

    methods = ["bootstrap", "gmm", "cvae"]
    

    for load_fn in datasets:
        data = load_fn()
        X = data["X"]
        y = data["y"]
        feature_names = data["feature_names"]

        p = X.shape[1]

        if feature_mode == "drop_one" and (drop_idx is None or drop_idx < 0 or drop_idx >= p):
            print(f"Skipping {data['dataset']} drop_idx={drop_idx}; p={p}")
            continue

        if feature_mode == "forward" and (k is None or k <= 0 or k > p):
            print(f"Skipping {data['dataset']} forward k={k}; p={p}")
            continue

        if feature_mode == "reverse" and (k is None or k < 0 or k >= p):
            print(f"Skipping {data['dataset']} reverse k={k}; p={p}")
            continue

        X_small, y_small, _ = stratified_subsample(
            X, y,
            n0=n0,
            n1=n1,
            seed=42
        )
        ranked_idx = rank_features_by_rf_importance(X_small, y_small, seed=42)
        keep = select_feature_subset(
            X_small,
            feature_names,
            mode=feature_mode,
            ranked_idx=ranked_idx,
            k=k,
            drop_idx=drop_idx
        )

        X_use = X_small[:, keep]
        feature_names_use = [feature_names[j] for j in keep]



## SAMPLING
        for method in methods:
            if method == "bootstrap":
                X_syn, y_syn = sample_bootstrap(X_use, y_small, n0, n1, seed=42)

            elif method == "gmm":
                X_syn, y_syn = sample_gmm(X_use, y_small, n0, n1, seed=42)

            elif method == "cvae":
                print("Training CVAE for", data["dataset"])
                best = train_cvae_on_arrays(
                    X_use,
                    y_small,
                    seed=42,
                    epochs=100,   
                    batch_size=32,

                )
                X_syn, y_syn = sample_cvae_dataset(best, n0, n1, seed=42)


        ## EVAL
            print(f"Evaluating {data['dataset']} | {method}")
            metrics, fig = evaluate_all(X_real=X_use, y_real= y_small,  X_syn=X_syn, y_syn = y_syn)

            if feature_mode == "full":
                n_features_used_label = f"{X_use.shape[1]}"
            elif feature_mode == "drop_one":
                n_features_used_label = f"{X_use.shape[1]}"
            elif feature_mode == "forward":
                n_features_used_label = f"top {k}"
            elif feature_mode == "reverse":
                n_features_used_label = f"drop top {k}"
            else:
                n_features_used_label = "unknown"

            row = {
                "dataset": data["dataset"],
                "category": data["category"],
                "n0": n0,
                "n1": n1,
                "method": method,
                "feature_mode": feature_mode,
                "k": None,
                "drop_idx": None,
                "n_features_used": n_features_used_label,
                "n_features_total": X.shape[1],
                "rf_auc_mean": metrics["rf_auc_mean"],
                "rf_auc_sd": metrics["rf_auc_sd"],
                "corr_mean_abs_diff": metrics["corr_mean_abs_diff"],
                "prop_significant": metrics["prop_significant"],
            }

            if feature_mode in {"forward", "reverse"}:
                row["k"] = k
            elif feature_mode == "drop_one":
                row["drop_idx"] = drop_idx

            rows.append(row)

            if figs is not None:
                figs.append({
                    "dataset": data["dataset"],
                    "method": method,
                    "fig": fig,
                })

            if isinstance(fig, dict):
                for f in fig.values():
                    plt.close(f)
            else:
                plt.close(fig)
    # df = pd.DataFrame(rows)
    # display(df)

# Cleaner Run

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def make_subset_metadata(feature_mode, p_total, keep, k=None, drop_idx=None):
    n_used = len(keep)

    if feature_mode == "full":
        return {
            "subset_family": "full",
            "subset_label": "all_features",
            "subset_param": np.nan,
            "n_features_used": n_used,
        }

    if feature_mode == "drop_one":
        return {
            "subset_family": "drop_one",
            "subset_label": f"drop_idx_{drop_idx}",
            "subset_param": drop_idx,
            "n_features_used": n_used,
        }

    if feature_mode == "forward":
        return {
            "subset_family": "forward",
            "subset_label": f"top_{k}",
            "subset_param": k,
            "n_features_used": n_used,
        }

    if feature_mode == "reverse":
        return {
            "subset_family": "reverse",
            "subset_label": f"drop_top_{k}",
            "subset_param": k,
            "n_features_used": n_used,
        }

    raise ValueError(f"Unknown feature_mode: {feature_mode}")


def run_experiment_clean(
    n0,
    n1,
    feature_mode="full",
    k=None,
    drop_idx=None,
    rows=None,
    figs=None,
    cvae_epochs=200,
    seed=42,
):
    if rows is None:
        rows = []

    datasets = [
        load_breast,
        load_diabetes,
    ]

    methods = ["bootstrap", "gmm", "cvae"]

    for load_fn in datasets:
        data = load_fn()
        X = data["X"]
        y = data["y"]
        feature_names = data["feature_names"]
        p = X.shape[1]

        # ---- guardrails ----
        if feature_mode == "drop_one" and (drop_idx is None or drop_idx < 0 or drop_idx >= p):
            print(f"Skipping {data['dataset']} drop_idx={drop_idx}; p={p}")
            continue

        if feature_mode == "forward" and (k is None or k <= 0 or k > p):
            print(f"Skipping {data['dataset']} forward k={k}; p={p}")
            continue

        if feature_mode == "reverse" and (k is None or k < 0 or k >= p):
            print(f"Skipping {data['dataset']} reverse k={k}; p={p}")
            continue

        # ---- balanced working sample ----
        X_small, y_small, _ = stratified_subsample(
            X, y,
            n0=n0,
            n1=n1,
            seed=seed,
        )

        ranked_idx = rank_features_by_rf_importance(X_small, y_small, seed=seed)

        keep = select_feature_subset(
            X_small,
            feature_names,
            mode=feature_mode,
            ranked_idx=ranked_idx,
            k=k,
            drop_idx=drop_idx,
        )

        X_use = X_small[:, keep]
        feature_names_use = [feature_names[j] for j in keep]

        subset_meta = make_subset_metadata(
            feature_mode=feature_mode,
            p_total=p,
            keep=keep,
            k=k,
            drop_idx=drop_idx,
        )

        # ---- generators ----
        for method in methods:
            if method == "bootstrap":
                X_syn, y_syn = sample_bootstrap(X_use, y_small, n0, n1, seed=seed)

            elif method == "gmm":
                X_syn, y_syn = sample_gmm(X_use, y_small, n0, n1, seed=seed)

            elif method == "cvae":
                print("Training CVAE for", data["dataset"])
                best = train_cvae_on_arrays(
                    X_use,
                    y_small,
                    seed=seed,
                    epochs=cvae_epochs,
                    batch_size=32,
                )
                X_syn, y_syn = sample_cvae_dataset(best, n0, n1, seed=seed)

            print(f"Evaluating {data['dataset']} | {method} | {subset_meta['subset_label']}")
            metrics, fig = evaluate_all(
                X_real=X_use,
                y_real=y_small,
                X_syn=X_syn,
                y_syn=y_syn,
            )

            row = {
                "dataset": data["dataset"],
                "category": data["category"],
                "method": method,
                "n0": n0,
                "n1": n1,
                "seed": seed,

                "feature_mode": feature_mode,
                "subset_family": subset_meta["subset_family"],
                "subset_label": subset_meta["subset_label"],
                "subset_param": subset_meta["subset_param"],
                "n_features_used": subset_meta["n_features_used"],
                "n_features_total": p,

                "rf_sep_mean": metrics.get("rf_sep_mean", np.nan),
                "rf_sep_sd": metrics.get("rf_sep_sd", np.nan),
                "rf_auc_mean": metrics.get("rf_auc_mean", np.nan),
                "rf_auc_sd": metrics.get("rf_auc_sd", np.nan),

                "corr_mean_abs_diff": metrics.get("corr_mean_abs_diff", np.nan),
                "corr_max_abs_diff": metrics.get("corr_max_abs_diff", np.nan),

                # keep if you want, but not primary
                "prop_significant": metrics.get("prop_significant", np.nan),

                # bookkeeping
                "kept_feature_idx": list(map(int, keep)),
                "kept_feature_names": feature_names_use,
            }

            rows.append(row)

            if figs is not None:
                figs.append({
                    "dataset": data["dataset"],
                    "method": method,
                    "feature_mode": feature_mode,
                    "subset_label": subset_meta["subset_label"],
                    "fig": fig,
                })

            if isinstance(fig, dict):
                for f in fig.values():
                    plt.close(f)
            else:
                plt.close(fig)

    return rows, figs

In [ ]:
rows = []
figures = []
run_experiment(n0=60, n1= 60,  feature_mode = "full", rows = rows, figs = figures)
df = pd.DataFrame(rows)
display(df)
print(len(figures))
print(figures[0].keys())

In [ ]:
figures[2]['fig']['pca']

In [ ]:
show_figures(figures, dataset="breast_cancer", method="bootstrap", plot="pca")

# Full Data Run
does synthetic quality depend on sample size?

In [ ]:
# rows = []

# for n0, n1 in [(23,68),(35,35),(50,50),(60,60)]:
#     run_experiment(
#         n0,
#         n1,
#         feature_mode="full",
#         rows=rows
#     )

# df = pd.DataFrame(rows)
# display(df)

# Drop One Test
Does one feature control the realism?

In [ ]:
# rows = []

# for j in range(30):
#     run_experiment(60,60,feature_mode="drop_one",drop_idx=j, rows = rows)

# df = pd.DataFrame(rows)
# display(df)

# Ablation

In [ ]:
# all_rows = []

# for k in [3, 5, 10, 15, 20]:
#     run_experiment(
#         n0=60,
#         n1=60,
#         feature_mode="forward",
#         k=k,
#         rows=all_rows
#     )

# for k in [1, 3, 5, 10]:
#     run_experiment(
#         n0=60,
#         n1=60,
#         feature_mode="reverse",
#         k=k,
#         rows=all_rows
#     )

# df_all = pd.DataFrame(all_rows)
# display(df_all)

In [ ]:
# summary_figs = evaluate_abl(df_all)

# Okay, evereything is tested. Please work.

In [ ]:
# all_rows = []
# figures = []
# # FULL 
# for n0, n1 in [(23,68),(35,35),(50,50),(60,60)]:
#     print(f"full: ({n0, n1})")
#     run_experiment(
#         n0,
#         n1,
#         feature_mode="full",
#         rows=all_rows,
#         figs = figures
#     )


# # DROP ONE 
# for j in range(30):
#     run_experiment(
#         60,
#         60,
#         feature_mode="drop_one",
#         drop_idx=j,
#         rows=all_rows,
#         figs = figures
#     )


# # FORWARD
# for k in [3, 5, 10, 15, 20]:
#     print(f"forward: {k}")
#     run_experiment(
#         n0=60,
#         n1=60,
#         feature_mode="forward",
#         k=k,
#         rows=all_rows,
#         figs = figures
#     )


# # REVERSE
# for k in [1, 3, 5, 10]:
#     print(f"reverse: {k}")
#     run_experiment(
#         n0=60,
#         n1=60,
#         feature_mode="reverse",
#         k=k,
#         rows=all_rows,
#         figs = figures
#     )


# df_all = pd.DataFrame(all_rows)

# print(df_all.shape)
# display(df_all.head())

In [ ]:
# df_all.to_csv("../results/synthetic_results_all.csv", index=False)

# Cleaner

In [ ]:
all_rows = []
all_figs = []

# 1) full-data runs across sample sizes
for n0, n1 in [(23, 68), (35, 35), (50, 50), (60, 60)]:
    print(f"FULL ({n0}, {n1})")
    run_experiment_clean(
        n0=n0,
        n1=n1,
        feature_mode="full",
        rows=all_rows,
        figs=all_figs,
        cvae_epochs=200,
        seed=42,
    )

# 2) drop-one only at fixed size
for j in range(30):
    run_experiment_clean(
        n0=60,
        n1=60,
        feature_mode="drop_one",
        drop_idx=j,
        rows=all_rows,
        figs=all_figs,
        cvae_epochs=200,
        seed=42,
    )

# 3) forward ablation
for k in [3, 5, 10, 15, 20]:
    print(f"FORWARD top {k}")
    run_experiment_clean(
        n0=60,
        n1=60,
        feature_mode="forward",
        k=k,
        rows=all_rows,
        figs=all_figs,
        cvae_epochs=200,
        seed=42,
    )

# 4) reverse ablation
for k in [1, 3, 5, 10]:
    print(f"REVERSE drop top {k}")
    run_experiment_clean(
        n0=60,
        n1=60,
        feature_mode="reverse",
        k=k,
        rows=all_rows,
        figs=all_figs,
        cvae_epochs=200,
        seed=42,
    )

df_all = pd.DataFrame(all_rows)
print(df_all.shape)
display(df_all.head())

# Export Figures

In [ ]:
import pickle
from pathlib import Path

OUTDIR = Path("../results")
OUTDIR.mkdir(parents=True, exist_ok=True)

# main clean CSV
df_all.to_csv(OUTDIR / "synthetic_results_clean.csv", index=False)

# summary ablation figures from your original helper
summary_figs = evaluate_abl(df_all)
with open(OUTDIR / "summary_figs.pkl", "wb") as f:
    pickle.dump(summary_figs, f)

# full saved diagnostic figures from evaluate_all(...)
with open(OUTDIR / "all_figs.pkl", "wb") as f:
    pickle.dump(all_figs, f)

print("Saved:")
print(OUTDIR / "synthetic_results_clean.csv")
print(OUTDIR / "summary_figs.pkl")
print(OUTDIR / "all_figs.pkl")